# Facebook100 walkthrough: Caltech

A compact tour of the `fb100` package on the smallest campus, Caltech (769 people). The
full 100-campus analysis is run by `make run`; its results are in `results/` and in the
[report](https://pchambet.github.io/facebook100-social-structure/).

Requires the data: `make data`.

In [1]:
from pathlib import Path

import pandas as pd

from fb100 import communities, homophily, labelprop, linkpred, structure
from fb100.io import ATTRIBUTES, load_school

DATA = Path("..") / "data" / "raw" / "facebook100"
school = load_school(DATA / "Caltech36.mat")
print(f"{school.name}: {school.n_nodes} nodes, {school.n_edges} edges")

Caltech36: 769 nodes, 16656 edges


## 1. Check the attribute columns before trusting any result

The dataset readme lists seven `local_info` columns: status, gender, major, minor, dorm,
year, high school. A quick look at the values confirms the order: the `year` column holds
calendar years and the `dorm` column holds eight codes, one per Caltech house. The first
version of this project skipped the minor column, so its "year" was in fact the dorm.

In [2]:
summary = pd.DataFrame(
    {
        "missing share": (school.attributes == 0).mean(),
        "distinct values": school.attributes.nunique(),
        "example values": [sorted(v[v != 0].unique())[:6] for _, v in school.attributes.items()],
    }
)
summary

,missing share,distinct values,example values
status,0.000000,4,"[1, 2, 5, 6]"
gender,0.085826,3,"[1, 2]"
major,0.100130,31,"[190, 192, 194, 195, 196, 197]"
minor,0.745124,36,"[190, 192, 194, 196, 197, 198]"
dorm,0.223667,9,"[165, 166, 167, 168, 169, 170]"
year,0.148244,18,"[1968, 1976, 1979, 1984, 1993, 1996]"
high_school,0.174252,501,"[250, 631, 636, 664, 719, 727]"


## 2. Structure

In [3]:
pd.Series(structure.summarize(school)).to_frame("Caltech36")

,Caltech36
school,Caltech36
nodes,769
edges,16656
mean_degree,43.318596
median_degree,36.0
density,0.056404
transitivity,0.291283
avg_clustering,0.409294
degree_assortativity,-0.065273
giant_component_share,0.990897


## 3. Homophily

Assortativity r per attribute, missing values excluded. At Caltech the dorm (house) effect
is the strongest of all, unlike most campuses where class year dominates.

In [4]:
r = {
    name: homophily.attribute_assortativity(school.adjacency, school.attributes[name].to_numpy())
    for name in ATTRIBUTES
}
pd.Series(r, name="r").sort_values(ascending=False).round(3)

dorm           0.449
year           0.269
status         0.244
gender         0.054
major          0.038
minor          0.004
high_school    0.002
Name: r, dtype: float64

## 4. Link prediction: easy benchmark versus full ranking

Hide 10% of the friendships, then score candidate pairs. The sampled AUC compares hidden
edges with random non-edges; R-precision ranks *every* unlinked pair.

In [5]:
rows = linkpred.evaluate(school.adjacency, seed=0, ks=(100,))
cols = ["method", "auc_sampled", "precision_at_100", "r_precision", "base_rate"]
pd.DataFrame(rows)[cols].round(4)

,method,auc_sampled,precision_at_100,r_precision,base_rate
0,common_neighbors,0.9381,0.5686,0.2621,0.0059
1,jaccard,0.9395,0.3800,0.2532,0.0059
2,adamic_adar,0.9425,0.6000,0.2707,0.0059
3,resource_allocation,0.9471,0.6500,0.2923,0.0059
4,preferential_attachment,0.8327,0.0800,0.0679,0.0059


## 5. Label propagation

Hide 20% of the known labels and recover them from the graph; compare with always
guessing the most frequent value.

In [6]:
lp = {
    name: labelprop.evaluate(school.adjacency, school.attributes[name].to_numpy(), 0.2, seed=0)
    for name in ["dorm", "year", "major", "gender"]
}
pd.DataFrame(lp).T[["accuracy", "majority_baseline", "classes", "hidden"]]

,accuracy,majority_baseline,classes,hidden
dorm,0.857143,0.176471,8.0,119.0
year,0.748092,0.198473,15.0,131.0
major,0.239130,0.108696,30.0,138.0
gender,0.617021,0.659574,2.0,141.0


## 6. Communities

Louvain communities against attributes (adjusted Rand index, 0 = chance). Caltech's
communities reproduce its houses, as Traud et al. (2012) found.

In [7]:
partition, modularity = communities.louvain(school.adjacency, seed=0)
print(f"{partition.max() + 1} communities, modularity {modularity:.3f}")
attrs = school.attributes[["dorm", "year", "major", "gender"]]
pd.DataFrame(communities.agreement(partition, attrs)).set_index("attribute").round(3)

11 communities, modularity 0.400


,ari,nmi
attribute,,
dorm,0.691,0.702
year,0.014,0.096
major,0.006,0.087
gender,0.002,0.010


In [8]:
# Cross-tabulation: share of each community living in its most common house
labelled = school.attributes["dorm"].to_numpy() != 0
table = pd.crosstab(partition[labelled], school.attributes["dorm"].to_numpy()[labelled])
(table.max(axis=1) / table.sum(axis=1)).round(2).rename("share in modal house").to_frame().T

row_0,1,2,3,4,5,6,7,8,9,10
share in modal house,0.93,0.94,0.29,0.89,1.0,0.52,0.5,0.5,0.89,0.89
